In [27]:
from pyspark.sql import SparkSession, DataFrame
from pyspark.ml import Pipeline
from pyspark.ml.feature import Imputer, StandardScaler, VectorAssembler
from pyspark.sql.functions import col, expr, lit
from functools import reduce
from dotenv import load_dotenv
import glob
import os

In [28]:
spark=SparkSession.builder.appName('Healthcare_DataCleaning').getOrCreate()

In [29]:
load_dotenv()

base_path=os.getenv('FOLDER_PATH')

folder_path = base_path+"/*.csv"
file_paths = glob.glob(folder_path)

In [30]:
file_paths=file_paths[:11]
file_paths

['D:/Coding_Projects/DataSets_health\\2013.csv',
 'D:/Coding_Projects/DataSets_health\\2014.csv',
 'D:/Coding_Projects/DataSets_health\\2015.csv',
 'D:/Coding_Projects/DataSets_health\\2016.csv',
 'D:/Coding_Projects/DataSets_health\\2017.csv',
 'D:/Coding_Projects/DataSets_health\\2018.csv',
 'D:/Coding_Projects/DataSets_health\\2019.csv',
 'D:/Coding_Projects/DataSets_health\\2020.csv',
 'D:/Coding_Projects/DataSets_health\\2021.csv',
 'D:/Coding_Projects/DataSets_health\\2022.csv',
 'D:/Coding_Projects/DataSets_health\\2023.csv']

In [31]:
sample_df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_paths[0])
print("Schema:")
sample_df.printSchema()
print("\nFirst 5 rows:")
sample_df.show(5)
print("\nColumn stats:")
sample_df.describe().show()

Schema:
root
 |-- Rndrng_Prvdr_Geo_Lvl: string (nullable = true)
 |-- Rndrng_Prvdr_Geo_Cd: string (nullable = true)
 |-- Rndrng_Prvdr_Geo_Desc: string (nullable = true)
 |-- HCPCS_Cd: string (nullable = true)
 |-- HCPCS_Desc: string (nullable = true)
 |-- HCPCS_Drug_Ind: string (nullable = true)
 |-- Place_Of_Srvc: string (nullable = true)
 |-- Tot_Rndrng_Prvdrs: integer (nullable = true)
 |-- Tot_Benes: integer (nullable = true)
 |-- Tot_Srvcs: double (nullable = true)
 |-- Tot_Bene_Day_Srvcs: integer (nullable = true)
 |-- Avg_Sbmtd_Chrg: double (nullable = true)
 |-- Avg_Mdcr_Alowd_Amt: double (nullable = true)
 |-- Avg_Mdcr_Pymt_Amt: double (nullable = true)
 |-- Avg_Mdcr_Stdzd_Amt: double (nullable = true)


First 5 rows:
+--------------------+-------------------+---------------------+--------+--------------------+--------------+-------------+-----------------+---------+---------+------------------+--------------+------------------+-----------------+------------------+
|Rndrng_Prv

In [32]:
target_cols = ['Rndrng_Prvdr_Geo_Cd','HCPCS_Cd','HCPCS_Drug_Ind','Place_Of_Srvc', 'Tot_Rndrng_Prvdrs','Tot_Benes','Tot_Srvcs','Tot_Bene_Day_Srvcs','Avg_Sbmtd_Chrg']

In [33]:
def clean_data(df_data):
    return (df_data.filter(col('Rndrng_Prvdr_Geo_Lvl') != 'National')
            .na.drop()
            .filter(~expr("try_cast(Rndrng_Prvdr_Geo_Cd as int)").isNull())
            .select(target_cols))

In [34]:
def get_year(file_path):
    year_part = os.path.basename(file_path).split('.')[0]
    return year_part

In [35]:
all_data = []
for file_path in file_paths:
    year = get_year(file_path)
    df = spark.read.option("header", "true").option("inferSchema", "true").csv(file_path)
    cleaned_df = clean_data(df)
    cleaned_df = cleaned_df.withColumn("Year", lit(year))
    all_data.append(cleaned_df)

    #making sure every data set is passed and cleaned
    print(file_path, "raw:", df.count(), "cleaned:", cleaned_df.count())

D:/Coding_Projects/DataSets_health\2013.csv raw: 264682 cleaned: 251600
D:/Coding_Projects/DataSets_health\2014.csv raw: 264758 cleaned: 251608
D:/Coding_Projects/DataSets_health\2015.csv raw: 266459 cleaned: 253084
D:/Coding_Projects/DataSets_health\2016.csv raw: 269836 cleaned: 256144
D:/Coding_Projects/DataSets_health\2017.csv raw: 269902 cleaned: 256139
D:/Coding_Projects/DataSets_health\2018.csv raw: 271742 cleaned: 257249
D:/Coding_Projects/DataSets_health\2019.csv raw: 273211 cleaned: 259317
D:/Coding_Projects/DataSets_health\2020.csv raw: 268149 cleaned: 254079
D:/Coding_Projects/DataSets_health\2021.csv raw: 271635 cleaned: 257325
D:/Coding_Projects/DataSets_health\2022.csv raw: 270673 cleaned: 256279
D:/Coding_Projects/DataSets_health\2023.csv raw: 268634 cleaned: 254097


In [36]:
final_df = reduce(DataFrame.unionByName, all_data)
final_df.groupBy("Year").count().orderBy("Year").show()
final_df.show(5)

+----+------+
|Year| count|
+----+------+
|2013|251600|
|2014|251608|
|2015|253084|
|2016|256144|
|2017|256139|
|2018|257249|
|2019|259317|
|2020|254079|
|2021|257325|
|2022|256279|
|2023|254097|
+----+------+

+-------------------+--------+--------------+-------------+-----------------+---------+---------+------------------+--------------+----+
|Rndrng_Prvdr_Geo_Cd|HCPCS_Cd|HCPCS_Drug_Ind|Place_Of_Srvc|Tot_Rndrng_Prvdrs|Tot_Benes|Tot_Srvcs|Tot_Bene_Day_Srvcs|Avg_Sbmtd_Chrg|Year|
+-------------------+--------+--------------+-------------+-----------------+---------+---------+------------------+--------------+----+
|                 01|   00100|             N|            F|              239|      219|    322.0|               322|  1069.9324224|2013|
|                 01|   00103|             N|            F|              515|     2394|   4206.0|              4206|  610.75801474|2013|
|                 01|   00104|             N|            F|              286|      309|   3103.0|       

In [37]:
final_df.toPandas().to_csv(base_path+'/final_data.csv', index=False)
print(f"Final cleaned data saved to {base_path}/final_data.csv")

Final cleaned data saved to D:/Coding_Projects/DataSets_health/final_data.csv
